In [59]:
import numpy as np
import pandas as pd

## 1. Load & Filter Data

In [60]:
df = pd.read_csv("../../../data/raw/PS_20174392719_1491204439457_log.csv")
print(len(df))
df.head()


6362620


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [61]:
print("Transaction Type that appears both Fraud and Not : ",df[df['isFraud']==1]['type'].unique())

Transaction Type that appears both Fraud and Not :  ['TRANSFER' 'CASH_OUT']


In [62]:
df = df[df['type'].isin(["CASH_OUT","TRANSFER"])]
print(len(df))
df.head()
# print(df.type.unique())

2770409


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
2,1,TRANSFER,181.00,C1305486145,181.0,0.0,C553264065,0.0,0.00,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.0,C38997010,21182.0,0.00,1,0
15,1,CASH_OUT,229133.94,C905080434,15325.0,0.0,C476402209,5083.0,51513.44,0,0
19,1,TRANSFER,215310.30,C1670993182,705.0,0.0,C1100439041,22425.0,0.00,0,0
24,1,TRANSFER,311685.89,C1984094095,10835.0,0.0,C932583850,6267.0,2719172.89,0,0


## 2. Handle Data Leakage (drop/adjust balance columns)

In [63]:
df.isnull().sum()

step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

## 3. Simple Feature Engineering (encoding type, is_new_destination)

## 4. Train-Test Split


In [64]:
from sklearn.model_selection import train_test_split

In [65]:
X = df.drop(columns = ["isFraud","isFlaggedFraud","nameOrig","nameDest"])
y = df.isFraud

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.3,stratify=y, random_state=42)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((1939286, 7), (831123, 7), (1939286,), (831123,))

In [67]:
X_train.isnull().sum()

step              0
type              0
amount            0
oldbalanceOrg     0
newbalanceOrig    0
oldbalanceDest    0
newbalanceDest    0
dtype: int64

In [68]:
X_train.head()

,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest
4788449,345,CASH_OUT,162481.30,160298.00,0.00,315690.93,478172.23
132994,11,CASH_OUT,452530.16,99634.00,0.00,0.00,2700525.59
6226235,590,CASH_OUT,157713.74,8020.00,0.00,40696.78,198410.51
3869441,283,CASH_OUT,6740.40,259729.75,252989.34,77942.78,84683.19
5436022,378,CASH_OUT,234698.82,0.00,0.00,5611307.47,5846006.29


In [69]:
# categorical_col = [col for col in X.columns if X[col].dtype == "object" or X[col].nunique() < 5]
# categorical_col

In [70]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler, RobustScaler,StandardScaler, PowerTransformer, OneHotEncoder
from sklearn.pipeline import Pipeline

In [71]:
cat_pipe = Pipeline([("ohe",  OneHotEncoder())])

preprocessor = ColumnTransformer([
    ("cat_pipe", cat_pipe, ["type"])
], remainder='passthrough')

In [72]:
from skopt import BayesSearchCV
from xgboost import XGBClassifier
from skopt.space import  Real,Integer

In [73]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
scale_pos_weight

336.3257957905723

In [74]:
param_distributions = {
    'algo__max_depth': Integer(low=1, high=10), 
    'algo__learning_rate': Real(low=1e-2, high=1.0, prior='log-uniform'),  
    'algo__n_estimators': Integer(low=100, high=200), 
    'algo__subsample': Real(low=0.3, high=0.8, prior='uniform'),
    'algo__gamma': Integer(low=1, high=10),
    'algo__colsample_bytree': Real(low=0.1, high=1.0, prior='uniform'),
    'algo__reg_alpha': Real(low=1e-3, high=10.0, prior='log-uniform'),   
    'algo__reg_lambda': Real(low=1e-3, high=10.0, prior='log-uniform'),
    'algo__scale_pos_weight': Real(1, scale_pos_weight , prior='log-uniform')
}

In [75]:
pipeline = Pipeline([
    ("prep", preprocessor),
    ("algo", XGBClassifier(n_jobs=-1,random_state=42, eval_metric='logloss'))
])
model = BayesSearchCV(
    estimator=pipeline,
    search_spaces=param_distributions,
    scoring='average_precision',  
    cv=5,
    n_iter=30,
    n_jobs=-1
)
model.fit(X_train,y_train)

d:\Kullyeah\Project\FraudDetection\.venv\Lib\site-packages\sklearn\compose\_column_transformer.py:1623: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


BayesSearchCV(cv=5,
              estimator=Pipeline(steps=[('prep',
                                         ColumnTransformer(remainder='passthrough',
                                                           transformers=[('cat_pipe',
                                                                          Pipeline(steps=[('ohe',
                                                                                           OneHotEncoder())]),
                                                                          ['type'])])),
                                        ('algo',
                                         XGBClassifier(base_score=None,
                                                       booster=None,
                                                       callbacks=None,
                                                       colsample_bylevel=None,
                                                       colsample_bynode=None,
                                                       colsample_bytree=None,
                                                       device=None,
                                                       early_stopping_rounds=None,
                                                       en...
                             'algo__reg_alpha': Real(low=0.001, high=10.0, prior='log-uniform', transform='normalize'),
                             'algo__reg_lambda': Real(low=0.001, high=10.0, prior='log-uniform', transform='normalize'),
                             'algo__scale_pos_weight': Real(low=1, high=336.3257957905723, prior='log-uniform', transform='normalize'),
                             'algo__subsample': Real(low=0.3, high=0.8, prior='uniform', transform='normalize')})

In [ ]:
# import xgboost as xgb
# from sklearn.metrics import average_precision_score

# scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
# print("Ideal scale_pos_weight:", scale_pos_weight)

# model_v2 = xgb.XGBClassifier(
#     scale_pos_weight=scale_pos_weight,
#     tree_method='hist',
#     n_estimators=200,
#     max_depth=6,
#     n_jobs=-1
# )
# model.fit(X_train, y_train)

# y_proba = model.predict_proba(X_test)[:, 1]
# print("PR-AUC (quick sanity check):", average_precision_score(y_test, y_proba))

Ideal scale_pos_weight: 336.3257957905723


KeyboardInterrupt: 

In [76]:
print(model.best_params_)
print(model.score(X_test,y_test), model.best_score_, model.score(X_train,y_train))

OrderedDict([('algo__colsample_bytree', 1.0), ('algo__gamma', 10), ('algo__learning_rate', 0.17448560131941038), ('algo__max_depth', 8), ('algo__n_estimators', 191), ('algo__reg_alpha', 2.3671400095732755), ('algo__reg_lambda', 10.0), ('algo__scale_pos_weight', 33.6583157400147), ('algo__subsample', 0.6256453527818211)])
0.9622487272743473 0.9628783873338401 0.9892764911793446


In [77]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = model.predict(X_test)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=4))

[[827594   1065]
 [    98   2366]]
              precision    recall  f1-score   support

           0     0.9999    0.9987    0.9993    828659
           1     0.6896    0.9602    0.8027      2464

    accuracy                         0.9986    831123
   macro avg     0.8447    0.9795    0.9010    831123
weighted avg     0.9990    0.9986    0.9987    831123



In [ ]:
best_model = model.best_estimator_

importances = best_model.named_steps['algo'].feature_importances_
feature_names = best_model.named_steps['prep'].get_feature_names_out()

importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)

print(importance_df.head(15))

                     feature  importance
5  remainder__newbalanceOrig    0.328845
7  remainder__newbalanceDest    0.234669
0    cat_pipe__type_CASH_OUT    0.210960
4   remainder__oldbalanceOrg    0.151381
3          remainder__amount    0.051165
2            remainder__step    0.014744
6  remainder__oldbalanceDest    0.008236
1    cat_pipe__type_TRANSFER    0.000000


## 5. Scaling (kalau perlu, misal buat Logistic Regression)

## 6. Train Baseline Model

## 7. Evaluate (precision, recall, F1, PR-AUC — BUKAN accuracy)